In [1]:
import pickle

# Definición de la instancia del grafo de Kanto para la simulación
f = frozenset

# Conjunto de vértices como frozenset
V = f({
    'Indigo Plateau',
    'Pallet Town',
    'Viridian City',
    'Pewter City',
    'Cinnabar Island',
    'Cerulean City',
    'Saffron City',
    'Celadon City',
    'Lavender Town',
    'Vermillion City',
    'Fuschia City',
    'Special Region'})

# Conjunto de aristas no dirigidas entre ciudades.
# Cada arista es un frozenset({u, v}) para que no importe el orden (u–v == v–u).
E = f({
    f({'Indigo Plateau', 'Viridian City'}),
    f({'Pallet Town', 'Viridian City'}),
    f({'Viridian City', 'Pewter City'}),
    f({'Pewter City', 'Cerulean City'}),
    f({'Cerulean City', 'Saffron City'}),
    f({'Saffron City', 'Celadon City'}),
    f({'Celadon City', 'Fuschia City'}),
    f({'Fuschia City', 'Special Region'}),
    f({'Fuschia City', 'Cinnabar Island'}),
    f({'Vermillion City', 'Special Region'}),
    f({'Cinnabar Island', 'Pallet Town'}),
    f({'Saffron City', 'Lavender Town'}),
    f({'Saffron City', 'Vermillion City'}),
    f({'Lavender Town', 'Cerulean City'}),
    f({'Lavender Town', 'Special Region'})
    })

# Tupla con el grafo en formato de conjuntos
G = (V, E)

# Usamos las listas, por comodidad
V = list(V) # lista de vertices
E = list(E) # lista de aristas ({u, v})


# Empaquetamos todo en 'data' y se guarda en un fichero binario con pickle
data = G, V, E
with open('instance.pickle', 'wb') as f:
    pickle.dump(data, f, pickle.HIGHEST_PROTOCOL)

print('Kanto League map problema successfully pickled!')

Kanto League map problema successfully pickled!


In [2]:
K = 2

origen = ["Pallet Town", "Fuschia City"]
destinos = ["Vermillion City", "Lavender Town"]

In [3]:
import pickle
import itertools
import sys
import os
from joblib import Parallel, delayed, parallel_backend
import importlib.metadata

project_path = "/home/pablo/Documentos/TFG/codigo_base/simulacion_adiabatica"
if project_path not in sys.path:
    sys.path.append(project_path)

import numpy as np
from qutip import basis, tensor, sesolve, Options, QobjEvo, expect, qeye, sigmaz
# basis(dim, i): |i> en dimensión 'dim'
# tensor(...): producto tensorial de kets/operadores
# sesolve: ecuación de Schrödinger
# QobjEvo: envoltorio para Hamiltonianos dependientes del tiempo H(t)
# expect: valor esperado <ψ|O|ψ>
# qeye: identidad
# sigmaz: matriz de Pauli Z

from qm import Sx, Sz

from hamiltonians import HMIS_, HMVC_

from solvers import solveAll

from utils import osum, makePairs

from collections import deque # Importamos la cola para usarla en BFS
import numpy as np

In [4]:
def bqm_add_linear(linear, i, c):
    """
    Añade/Acumula un término lineal al QUBO.

    Modifica el diccionario `linear` in-place sumando:
        linear[i] += c

    Params
    ----------
    linear : dict[int, float]
        Diccionario de términos lineales del QUBO.

    i : int
        Índice de variable binaria.

    c : float
        Coeficiente a sumar.
    """
    linear[i] = linear.get(i, 0.0) + float(c)

def bqm_add_quad(quadratic, i, j, c):
    """
    Añade/Acumula un término cuadrático al QUBO (interacción entre variables i y j).

    Modifica `quadratic` in-place sumando:
        quadratic[(min(i,j), max(i,j))] += c

    Params
    ----------
    quadratic : dict[tuple[int,int], float]
        Diccionario de términos cuadráticos.

    i, j : int
        Índices de variables binarias (deben ser distintos).

    c : float
        Coeficiente a sumar.
    """
    if i == j:
        raise ValueError("No pongas términos (i,i) en quadratic; intégralos en linear si hace falta.")
    i, j = (i, j) if i < j else (j, i)
    quadratic[(i, j)] = quadratic.get((i, j), 0.0) + float(c)

def bqm_add_offset(current_offset, c):
    """
    Aumenta el término constante (offset) del QUBO.

    Params
    ----------
    current_offset : float
        Offset actual.

    c : float
        Incremento.

    Returns
    -------
    float
        Nuevo offset = current_offset + c
    """
    return current_offset + float(c)

def exactamente_uno(linear, quadratic, offset, vars_set, P):
    """
    Impone la restricción one-hot: EXACTAMENTE 1 variable activa en `vars_set`.

    Añade la penalización:
        P * (1 - Σ x_k)^2

    Expansión (x binaria, x^2=x):
      - offset += P
      - lineal  += -P * x_i
      - cuadrático += 2P * x_i x_j  (i<j)

    Params
    ----------
    linear, quadratic, offset :
        Estructura del QUBO a modificar.
    vars_set : list[int]
        Conjunto/lista de índices de variables.
    P : float
        Penalización (debe ser suficientemente grande para dominar a la parte de coste).

    Return
    -------
    float
        Nuevo offset tras aplicar la penalización.
    """
    # Se expande el binomio (1 - sum x)^2 = 1 - 2*sum(x) + sum(x)^2
    # Como x^2 = x (para binarios), se simplifica algebraicamente.
    offset = bqm_add_offset(offset, P)
    for i in vars_set:
        bqm_add_linear(linear, i, -P)
    for indx_a in range(len(vars_set)):
        for indx_b in range(indx_a+1, len(vars_set)):
            a, b = vars_set[indx_a], vars_set[indx_b]
            bqm_add_quad(quadratic, a, b, 2.0*P) # Penaliza tener dos qubits activos simultáneamente
    return offset

def exactamente_k(linear, quadratic, offset, vars_set, K, P):
    """
    Impone EXACTAMENTE K variables activas en `vars_set`.

    Penalización:
        P * (Σ x_i - K)^2

    Expansión (x binaria, x^2=x):
      - offset      += P*K^2
      - lineales    += P*(1 - 2K) * x_i
      - cuadráticos += 2P * x_i x_j

    Params
    ----------
    linear, quadratic, offset :
        Estructura del QUBO a modificar.
    vars_set : list[int]
        Conjunto/lista de índices.
    K : int
        Número exacto de activos.
    P : float
        Penalización.

    Returns
    -------
    float
        Nuevo offset tras aplicar la penalización.
    """
    
    offset = bqm_add_offset(offset, P * (K ** 2))
    for i in vars_set:
        bqm_add_linear(linear, i, P * (1.0 - 2.0 * K))
    for a in range(len(vars_set)):
        for b in range(a + 1, len(vars_set)):
            bqm_add_quad(quadratic, vars_set[a], vars_set[b], 2.0 * P)
    return offset

def dist_bfs(stqubo):
    """
    Calcula distancias mínimas desde un nodo origen usando BFS (en número de aristas).

    Params
    ----------
    stqubo : hashable
        Nodo origen.

    Returns
    -------
    dict
        Diccionario dist[u] = distancia mínima desde stqubo a u.
        Si un nodo no es alcanzable, su distancia queda como np.inf.
    """
    dist = {u: np.inf for u in V}
    dist[stqubo] = 0
    q = deque([stqubo])
    while q:
        u = q.popleft()
        for w in adj[u]:
            if dist[w] == np.inf:
                dist[w] = dist[u] + 1
                q.append(w)
    return dist

def indx(k, i, t, n_ciudades, n_slots):
    """
    Mapea una variable 2D x_{k,i,t} (ciudad i, slot t) a un índice 1D único.
    Se usa para aplanar una matriz (n_ciudades x n_slots) en un vector de qubits.

    Params
    ----------
    i : int
        Índice de ciudad.

    t : int
        Índice de slot/tiempo.

    n_slots : int
        Número total de slots.

    Returns
    -------
    int
        Índice único: i*n_slots + t
    """
    return k * (n_ciudades * n_slots) + i * n_slots + t  # Aplana matriz (ciudades x tiempo) a vector 1D

def mvrp_qubo(origenes, destinos, K, dists, P_pos, P_uno, P_deposito):
    """
    Construye el QUBO para un MVRP (multi-depósito, multi-vehículo) codificado por slots.

    Encoding:
      - Variables x_{i,t} ∈ {0,1} indican ciudad i elegida en el slot t.
      - One-hot por slot: en cada t exactamente una ciudad.
      - Cada destino aparece exactamente una vez total.
      - Depósitos pueden repetirse.
      - Condición de frontera: slot 0 debe ser el depósito 0 (en esta versión).
      - Último slot: debe ser un depósito (cualquiera).
      - Penalización por repetir ciudad en slots consecutivos (evitar quedarse quieto).

    Params
    ----------
    origenes : list[str] | str
        Depósitos (si es str se convierte a lista de un elemento).

    destinos : list[str]
        Ciudades destino.

    K : int
        Número de vehículos (interpretado como “número de apariciones de depósito” en el encoding).
        Nota: el sentido exacto de K depende de tu segmentación posterior.

    dists : dict[str, dict[str, float]]
        Distancias completas entre ciudadea.

    P_pos : float
        Penalización para la restricción one-hot por slot.

    P_uno : float
        Penalización para “cada destino exactamente una vez”.

    P_deposito : float
        Penalización para condiciones de depósito (inicio/fin y/o continuidad).

    Returns
    -------
    dict
        Diccionario con:
          - linear, quadratic, offset : QUBO en forma de diccionarios
          - N : número de variables (qubits)
          - K : K usado
          - ciudades : lista completa (depósitos + destinos)
          - n_slots : nº slots
          - origenes_indx : lista de índices de depósitos
          - costes : matriz NxN de costes
          - varmap : mapa índice->(ciudad,slot)
    """

    if isinstance(origenes, str):
        origenes_aux = [origenes]
    else:
        origenes_aux = list(origenes)
    
    ciudades = origenes + destinos
    n_origenes = len(origenes_aux)
    n_ciudades = len(ciudades)
    n_destinos = len(destinos)
    # Slots necesarios: 1 por destino + 1 por inicio de vehículo + 1 cierre final
    n_slots = n_destinos + 2

    qubo_lineal, qubo_quad, offset_bqm = {}, {}, 0.0

    # Construcción de la matriz de costes (grafo completo con distancias)
    costes = np.zeros((n_ciudades, n_ciudades), dtype=float)
    for i, ci in enumerate(ciudades):
        for j, cj in enumerate(ciudades):
            costes[i, j] = float(dists[ci][cj])

    # Check de seguridad: La penalización P debe ser mayor que cualquier beneficio por distancia
    max_coste_mapa = np.max(costes)
    if P_pos < max_coste_mapa:
        print(f"AVISO: Aumentando P_pos de {P_pos} a {max_coste_mapa * 2} para evitar errores lógicos.")
        P_pos = max_coste_mapa * 2
        P_uno = P_pos + 10
        P_deposito = P_pos

    # 1. FUNCIÓN OBJETIVO: Minimizar distancia recorrida
    # Suma coste[i,j] si en t estamos en i y en t+1 estamos en j
    for k in range(K):
        for t in range(n_slots - 1):
            t_next = t + 1
            for i in range(n_ciudades):
                for j in range(n_ciudades):
                    coste = costes[i, j]
                    if coste != 0.0:
                        bqm_add_quad(
                            qubo_quad,
                            indx(k, i, t, n_ciudades, n_slots),
                            indx(k, j, t_next, n_ciudades, n_slots),
                            coste
                        )

    # 2. RESTRICCIÓN: En cada instante t, el vehículo debe estar en EXACTAMENTE UNA ciudad
    for k in range(K):
        for t in range(n_slots):
            S = [indx(k, i, t, n_ciudades, n_slots) for i in range(n_ciudades)]
            offset_bqm = exactamente_uno(qubo_lineal, qubo_quad, offset_bqm, S, P_pos)

    # 3. RESTRICCIÓN: Cada ciudad destino debe ser visitada EXACTAMENTE UNA vez en total
    # (Los depósitos se pueden visitar múltiples veces, por eso range empieza en n_origenes)
    for i in range(n_origenes, n_ciudades):
        S = [
            indx(k, i, t, n_ciudades, n_slots)
            for k in range(K)
            for t in range(n_slots)
        ]
        offset_bqm = exactamente_k(qubo_lineal, qubo_quad, offset_bqm, S, 1, P_uno)

    # 4. CONDICIÓN DE FRONTERA: En t=0, debe estar el primer depósito
    for k in range(K):
        origen_k = k   # ejemplo simple: vehículo k sale del origen k
        bqm_add_linear(qubo_lineal, indx(k, origen_k, 0, n_ciudades, n_slots), -P_deposito)
        offset_bqm = bqm_add_offset(offset_bqm, P_deposito)

    # 5. CONTINUIDAD DE DEPÓSITO: Si está en un depósito en t, no debe de quedarse
    # Se penaliza ligeramente el cambio entre depósitos para evitar saltos innecesarios si no es para ruta
    for k in range(K):
        for t in range(n_slots - 1):
            for d1 in range(n_origenes):
                for d2 in range(n_origenes):
                    bqm_add_quad(
                        qubo_quad,
                        indx(k, d1, t, n_ciudades, n_slots),
                        indx(k, d2, t + 1, n_ciudades, n_slots),
                        P_deposito * 0.5
                    )

    # 6. CONDICIÓN DE FRONTERA FINAL: En el último slot, debe terminar en un depósito
    for k in range(K):
        SL = [indx(k, d, n_slots - 1, n_ciudades, n_slots) for d in range(n_origenes)]
        offset_bqm = exactamente_uno(qubo_lineal, qubo_quad, offset_bqm, SL, P_deposito)

    # 7. CONTINUIDAD DE CIUDAD: no debe quedarse en una misma ciudad, esto se penaliza fuertemente
    P_rep = P_pos  # o 2*max_coste_mapa
    for k in range(K):
        for t in range(n_slots - 1):
            for i in range(n_ciudades):
                bqm_add_quad(
                    qubo_quad,
                    indx(k, i, t, n_ciudades, n_slots),
                    indx(k, i, t+1, n_ciudades, n_slots),
                    P_rep
                )

    N_qubits = K * n_ciudades * n_slots
    varmap = {indx(k, i, t, n_ciudades, n_slots): (k, i, t) for k in range(K) for i in range(n_ciudades) for t in range(n_slots)}

    return {
        "linear": qubo_lineal,
        "quadratic": qubo_quad,
        "offset": float(offset_bqm),
        "N": int(N_qubits),
        "K": int(K),
        "ciudades": ciudades,
        "n_slots": int(n_slots),
        "origenes_indx": list(range(n_origenes)),
        "costes": costes,
        "n_destinos": n_destinos,
        "varmap": varmap
    }

In [5]:
P = 50.0
qubo_lineal = {}
qubo_quad = {}
offset_bqm = 0.0
adj = {v: set() for v in V}
for e in E:
    a, b = tuple(e)
    adj[a].add(b); adj[b].add(a)
dists = {v: dist_bfs(v) for v in V}
qubo = mvrp_qubo(origen, destinos, K, dists, P_pos=P, P_uno=P + 10, P_deposito=P)

N_qubits = qubo["N"]
n_slots  = qubo["n_slots"]
n_destinos = qubo["n_destinos"]

N = N_qubits

print(f"MVRP: depósito = {origen}")
print(f"MVRP: destinos = {destinos}  (n_destinos={n_destinos})")
print(f"MVRP: vehículos K = {K}, slots = {n_slots}, N_qubits = {N_qubits}")

MVRP: depósito = ['Pallet Town', 'Fuschia City']
MVRP: destinos = ['Vermillion City', 'Lavender Town']  (n_destinos=2)
MVRP: vehículos K = 2, slots = 4, N_qubits = 32


In [6]:
import os, json

os.makedirs("pruebas_extra", exist_ok=True)

path_json = "pruebas_extra/mvrp_qubo_DistintoOrigen32qubits.json"

qubo_json = {
    "linear": {str(k): float(v) for k, v in qubo["linear"].items()},
    "quadratic": {f"{i},{j}": float(c) for (i, j), c in qubo["quadratic"].items()},
    "offset": float(qubo["offset"]),
    "N": int(qubo["N"]),
    "K": int(qubo["K"]),
    "ciudades": qubo["ciudades"],
    "n_slots": int(qubo["n_slots"]),
    "origenes_indx": list(qubo["origenes_indx"]),
    "varmap": {str(idx): [int(k), int(i), int(t)] for idx, (k, i, t) in qubo["varmap"].items()}
}

with open(path_json, "w", encoding="utf-8") as f:
    json.dump(qubo_json, f, indent=2, ensure_ascii=False)